# 01 Sample construction and abnormal returns

Builds the event sample (Chapter 3) and the FF5 abnormal returns (Chapter 4).

1. Link each Bloomberg event to a CRSP permno or Datastream infocode.
2. Keep events whose firm is an S&P 500 / STOXX Europe 600 member on the event date.
3. Estimate the FF5 model over $\tau \in \{-250, \ldots, -20\}$ and compute $AR_{i,k}$ for $k=-5,\dots,+5$.
4. Drop events within $\pm 5$ calendar days of an earnings announcement.
5. Flag Third Bridge calls, firm-hosted events and conference-day clusters.

Output: `intermediate/events_final.parquet` in the output folder, the final sample of 615 events read by notebooks 02 to 05.

In [1]:
import re

import numpy as np
import pandas as pd

from src.common import (input_path, EVENT_COLUMNS, INTERIM, TABLES, EST_LEN, EST_GAP, MIN_OBS, FCOLS,
                        K_VALS, ARC, DATEC, GICS_TO_BROAD, ar_col, date_col, lev_col,
                        load_constituents, load_returns, series_by_id, load_factors,
                        constituent_maps, resolve_identifier)

## Events

The 685 AI-related calendar entries that pass the relevance check and the attribution, identifier and
return-data filters (Table 3.1), before the index-membership and earnings filters below. A record that names
several firms is already assigned to the first-listed index constituent with return data (Section 3.1.2).
`Event_Type` holds the Bloomberg code (MA, CP, CM or CS). The seven ID, SM and TL events are pooled under the
label `Other`, which differs from Bloomberg's Other category (MA and TL, Section 3.1.4).

In [2]:
events = pd.read_excel(input_path("events"))[EVENT_COLUMNS]
events["Date"] = pd.to_datetime(events["Date"]).dt.normalize()
events["Verified_Results_Date"] = pd.to_datetime(events["Verified_Results_Date"])
events["base"] = events["Ticker"].str.split().str[0]
events["event_id"] = events["Ticker"] + "|" + events["Date"].dt.strftime("%Y-%m-%d")
assert not events["event_id"].duplicated().any()
assert events["Event_Type"].isin(["MA", "CP", "CM", "CS", "Other"]).all()
print(f"{len(events)} candidate events (US {(events['Region'] == 'US').sum()}, "
      f"EU {(events['Region'] == 'EU').sum()})")

685 candidate events (US 603, EU 82)


## Returns, factors and index membership

Membership end dates: CRSP marks current constituents with an end date of 31 December 2025, the last date it
covers, and Datastream with a placeholder date. The `still_active` flag treats both as open, so the 2026 US
events use each firm's status on 31 December 2025 (Section 3.2).

In [3]:
us_c, eu_c = load_constituents()
us_map, eu_map, ambiguous = constituent_maps(us_c, eu_c)
us_sector = us_c.drop_duplicates("permno").set_index("permno")[["gics_sector_code", "sector_available"]]
eu_sector = eu_c.drop_duplicates("infocode").set_index("infocode")[
    ["gics_sector_code", "sector_available", "sector_source"]]

us_r, eu_r = load_returns()
ret = {"US": series_by_id(us_r, "permno", "daily_return"),
       "EU": series_by_id(eu_r, "infocode", "ret_usd")}
mcap = {"US": series_by_id(us_r, "permno", "market_cap"),
        "EU": series_by_id(eu_r, "infocode", "mktcap_usd")}
FF = load_factors()

FAR_FUTURE = pd.Timestamp("2100-01-01")


def member_spells(c, key):
    end = c["membership_end"].where(~c["still_active"], FAR_FUTURE)
    spells = {}
    for k, s, e in zip(c[key], c["membership_start"], end):
        spells.setdefault(k, []).append((s, e))
    return spells


spells = {"US": member_spells(us_c, "permno"), "EU": member_spells(eu_c, "infocode")}


def is_member(ident, region, d):
    if ident is None:
        return None
    return any(s <= d <= e for s, e in spells[region].get(ident, []))


def find_returns(ticker, region, ds_ticker, d):
    ident, reg = resolve_identifier(ticker, region, ds_ticker, us_map, eu_map, ambiguous)
    if ident is None or ident not in ret[reg]:
        return None, None, None, None
    s = ret[reg][ident]
    if len(s[s.index <= d]) > EST_LEN and len(s[s.index > d]) >= 5:
        return s, mcap[reg].get(ident), reg, ident
    return None, None, None, None

## FF5 estimation

$R_{it} - R_{ft} = \alpha_i + \beta_i' f_t + \varepsilon_{it}$ over $\tau \in \{-250, \ldots, -20\}$.
An event on a non-trading day moves to the next trading day (at most 3 calendar days).
`Lev_*` is the forecast-error term $x_w'(X'X)^{-1}x_w$ of each window, used in the denominator of $SCAR_i$
(Equation 4.8). The estimation residuals are saved for the $\bar\rho$ estimate in the KP test.

In [4]:
LEVERAGE_WINDOWS = [(0, 0), (-1, 1), (0, 1), (0, 3), (-2, 2), (-3, 3), (-4, 4), (-5, 5)]
LEVC = [lev_col(k1, k2) for k1, k2 in LEVERAGE_WINDOWS]

rows, skipped, link_rows, resid_rows = [], {}, [], []
for _, e in events.iterrows():
    s, mc, reg, ident = find_returns(e["Ticker"], e["Region"], e["Datastream_Ticker"], e["Date"])
    member = is_member(ident, reg, e["Date"])
    link_rows.append(dict(Ticker=e["Ticker"], Region=e["Region"], matched=s is not None,
                          source={"US": "CRSP+DS_fallback", "EU": "Datastream"}.get(reg),
                          identifier=ident, member_on_event_date=member))
    if s is None:
        skipped["no_match"] = skipped.get("no_match", 0) + 1
        continue
    if not member:
        skipped["not_index_member"] = skipped.get("not_index_member", 0) + 1
        continue

    sec = (us_sector if reg == "US" else eu_sector)
    sec = sec.loc[ident] if ident in sec.index else None
    avail = sec is not None and sec["sector_available"]
    sector = GICS_TO_BROAD.get(str(sec["gics_sector_code"])) if avail else None
    sector_source = (("ISIN" if reg == "US" else sec["sector_source"]) if avail else None)

    mktcap = None
    if mc is not None:
        prior = mc[mc.index <= e["Date"]]
        if len(prior):
            mktcap = round(float(prior.iloc[-1]) / 1e6, 2)

    h = s.to_frame("r").join(FF[reg], how="inner").dropna()
    d = e["Date"]
    if d not in h.index:
        nxt = h.index[h.index >= d]
        if len(nxt) == 0 or (nxt[0] - d).days > 3:
            skipped["no_coverage"] = skipped.get("no_coverage", 0) + 1
            continue
        d = nxt[0]
    i0 = h.index.get_loc(d)
    if i0 < EST_LEN + 1 or i0 + 5 >= len(h):
        skipped["window_out_of_range"] = skipped.get("window_out_of_range", 0) + 1
        continue
    est = h.iloc[i0 - EST_LEN: i0 - EST_GAP + 1]
    if len(est) < MIN_OBS:
        skipped["short_estimation"] = skipped.get("short_estimation", 0) + 1
        continue
    X = np.column_stack([np.ones(len(est)), est[FCOLS].to_numpy(float)])
    y = (est["r"] - est["RF"]).to_numpy(float)
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ b
    sigma = resid.std(ddof=6)
    if sigma <= 0:
        skipped["degenerate_sigma"] = skipped.get("degenerate_sigma", 0) + 1
        continue
    r2 = 1 - (resid ** 2).sum() / ((y - y.mean()) ** 2).sum()

    XtX_inv = np.linalg.pinv(X.T @ X)
    lev = {}
    for k1, k2 in LEVERAGE_WINDOWS:
        xsum = np.concatenate([[k2 - k1 + 1],
                               h.iloc[i0 + k1: i0 + k2 + 1][FCOLS].to_numpy(float).sum(axis=0)])
        lev[lev_col(k1, k2)] = float(xsum @ XtX_inv @ xsum)

    for dt, rv in zip(est.index, resid):
        resid_rows.append(dict(Ticker=e["Ticker"], Date=e["Date"], Region=reg,
                               est_date=dt, resid=float(rv)))

    ar, ar_date = {}, {}
    for k in K_VALS:
        z = h.iloc[i0 + k]
        ar[ar_col(k)] = float((z["r"] - z["RF"] - b[0] - np.dot(z[FCOLS].to_numpy(float), b[1:])) * 100)
        ar_date[date_col(k)] = z.name
    rows.append(dict(Ticker=e["Ticker"], Date=e["Date"], source=link_rows[-1]["source"],
                     Identifier=ident, Sector=sector, Sector_source=sector_source,
                     MktCap_USDm=mktcap, Sigma=round(sigma, 4), R2=round(r2, 4),
                     **ar, **ar_date, **lev))

est_df = pd.DataFrame(rows)
est_df["CAR_1"] = est_df[[ar_col(-1), ar_col(0), ar_col(1)]].sum(axis=1)
print(f"Estimated: {len(est_df)} of {len(events)}; excluded: {skipped}")

events_all = events.drop(columns=["base"]).merge(est_df, on=["Ticker", "Date"], how="left",
                                                 validate="one_to_one")
events_all = events_all[["event_id", "Company", "Ticker", "Date", "Region", "Event_Type", "Description",
                         "Record_Firms", "Firm_Hosted", "IBES_Name", "Verified_Results_Date", "Sector",
                         "Sector_source", "Identifier", "MktCap_USDm", "source",
                         "Sigma", "R2", "CAR_1"] + ARC + DATEC + LEVC]
events_all.to_parquet(INTERIM / "events_all.parquet", index=False)
pd.DataFrame(link_rows).to_csv(INTERIM / "event_linkage.csv", index=False)
pd.DataFrame(resid_rows).to_parquet(INTERIM / "estimation_residuals.parquet", index=False)

Estimated: 666 of 685; excluded: {'not_index_member': 19}


## Earnings screen

An event is contaminated if an earnings announcement falls within $\pm 5$ calendar days. US firms link to
I/B/E/S by CUSIP, European firms by the I/B/E/S company name in `IBES_Name`. Where I/B/E/S cannot screen
an event (results date after its last record, or firm not covered), `Verified_Results_Date` is used, and
every such event must have one.

In [5]:
WINDOW_DAYS = 5
ib_us = pd.read_parquet(input_path("earnings_us"))
ib_eu = pd.read_parquet(input_path("earnings_intl"))
for ib in (ib_us, ib_eu):
    ib["anndats"] = pd.to_datetime(ib["anndats"], errors="coerce")
ibes_max = min(ib_us["anndats"].max(), ib_eu["anndats"].max())

cus = (us_c[["permno", "cusip"]].dropna().drop_duplicates()
       .assign(cusip8=lambda d: d["cusip"].astype(str).str[:8]))
permno_to_cusips = cus.groupby("permno")["cusip8"].apply(set).to_dict()
ib_us["cusip8"] = ib_us["cusip"].astype(str).str[:8]
us_dates = ib_us.dropna(subset=["anndats"]).groupby("cusip8")["anndats"].apply(lambda s: np.sort(s.unique()))


def norm(s):
    s = re.sub(r"[^A-Z0-9 ]", " ", str(s).upper())
    return re.sub(r"\s+", " ", s).strip()


ib_eu["cname_n"] = ib_eu["cname"].map(norm)
eu_dates = ib_eu.dropna(subset=["anndats"]).groupby("cname_n")["anndats"].apply(lambda s: np.sort(s.unique()))

screen, unscreened = [], []
for _, r in events_all.iterrows():
    base = dict(event_id=r["event_id"], Company=r["Company"], Ticker=r["Ticker"],
                Date=r["Date"].date(), Region=r["Region"], CAR_1=r["CAR_1"])
    if pd.isna(r["Sigma"]) or pd.isna(r["Identifier"]):
        screen.append(dict(**base, status="membership_dropped", source=None, results_date=None, day_diff=np.nan))
        continue
    if r["Region"] == "US":
        keys = permno_to_cusips.get(r["Identifier"], set())
        hits = [d for c in keys for d in us_dates.get(c, [])]
    else:
        keys = [] if pd.isna(r["IBES_Name"]) else str(r["IBES_Name"]).split("|")
        hits = [d for c in keys for d in eu_dates.get(c, [])]
    dates = np.sort(np.unique(hits)) if hits else np.array([])

    results, diff, status, source = pd.NaT, np.nan, None, "I/B/E/S"
    if len(dates):
        diffs = (pd.DatetimeIndex(dates) - r["Date"]).days
        i = int(np.argmin(np.abs(diffs)))
        results, diff = pd.Timestamp(dates[i]), int(diffs[i])
        if abs(diff) <= WINDOW_DAYS:
            status = "contaminated"
        elif (ibes_max - r["Date"]).days >= WINDOW_DAYS:
            status = "clean"
    if status is None:  # not covered by I/B/E/S, or too close to its last record
        if pd.isna(r["Verified_Results_Date"]):
            unscreened.append(r["event_id"])
            continue
        results, source = r["Verified_Results_Date"], "results calendar"
        diff = int((results - r["Date"]).days)
        status = "contaminated" if abs(diff) <= WINDOW_DAYS else "clean"
    elif pd.notna(r["Verified_Results_Date"]):
        raise SystemExit(f"{r['event_id']}: verified results date given, but I/B/E/S screens this event")
    screen.append(dict(**base, status=status, source=source, results_date=results.date(), day_diff=diff))
if unscreened:
    raise SystemExit(f"No earnings date for: {sorted(unscreened)}")
screen = pd.DataFrame(screen)

screen.to_csv(INTERIM / "earnings_screen_report.csv", index=False)
exclusions = screen[screen["status"] == "contaminated"]
exclusions.to_csv(INTERIM / "earnings_exclusions.csv", index=False)
print(f"I/B/E/S coverage ends {ibes_max.date()}")
print(screen.groupby(["status", "source"], dropna=False).size().to_string())

I/B/E/S coverage ends 2026-05-14
status              source          
clean               I/B/E/S             612
                    results calendar      3
contaminated        I/B/E/S              50
                    results calendar      1
membership_dropped  NaN                  19


## Final sample and event flags

- **Third Bridge**: expert-network calls hosted by Third Bridge, in which an industry specialist discusses a
  company or sector for a subscriber audience (Section 3.1.4). They are identified from the calendar description.
- **Firm-hosted**: entries hosted by the tagged firm itself, classified by hand from the description and the
  webcast host (`Firm_Hosted`, Section 3.1.4).
- **Tier**: firm-hosted, conference appearance (the other CP events), Third Bridge call, other
  research-provider or broker session (CM, CS), and the two residual events (`Other`).
- **Cluster days**: the ten dates with the most events (ties broken by earlier date).

In [6]:
df = events_all[events_all["Sigma"].notna()]
df = df[~df["event_id"].isin(set(exclusions["event_id"]))].copy()
df["base"] = df["Ticker"].str.split().str[0]
df["firm_key"] = df["Region"] + "|" + df["Identifier"].astype(str)
df["EU"] = (df["Region"] == "EU").astype(int)

df["third_bridge"] = df["Description"].fillna("").str.contains("Third Bridge", case=False, regex=False)
df["source_record_nfirms"] = df["Record_Firms"]
df["firm_hosted"] = df["Firm_Hosted"] == 1
df = df.drop(columns=["Description", "Record_Firms", "Firm_Hosted", "IBES_Name", "Verified_Results_Date"])

df["tier"] = np.select(
    [df["firm_hosted"], df["third_bridge"], df["Event_Type"] == "CP", df["Event_Type"].isin(["CM", "CS"])],
    ["Firm-hosted", "Third Bridge", "Conference appearance", "Research-provider or broker session"],
    default="Other")

day_counts = (df.groupby("Date").size().rename("n").reset_index()
              .sort_values(["n", "Date"], ascending=[False, True]).set_index("Date")["n"])
top10 = day_counts.head(10)
df["on_cluster_day"] = df["Date"].isin(set(top10.index)).astype(int)
cluster_days = top10.rename("n_events").reset_index()
cluster_days["share_CP"] = cluster_days["Date"].map(lambda d: (df[df["Date"] == d]["Event_Type"] == "CP").mean())
cluster_days.to_csv(TABLES / "cluster_days.csv", index=False)

df.to_parquet(INTERIM / "events_final.parquet", index=False)
print(f"Final sample: {len(df)} events (US {(df['Region'] == 'US').sum()}, EU {df['EU'].sum()}), "
      f"{df['firm_key'].nunique()} firms")
print(f"Third Bridge {df['third_bridge'].sum()}, firm-hosted {df['firm_hosted'].sum()}")
print("Source record size: 1 firm {}, 2-7 {}, 8+ {}".format(
    (df["source_record_nfirms"] == 1).sum(), df["source_record_nfirms"].between(2, 7).sum(),
    (df["source_record_nfirms"] >= 8).sum()))
print(df["tier"].value_counts().to_string())

Final sample: 615 events (US 545, EU 70), 174 firms
Third Bridge 198, firm-hosted 42
Source record size: 1 firm 333, 2-7 192, 8+ 90
tier
Conference appearance                  248
Third Bridge                           198
Research-provider or broker session    125
Firm-hosted                             42
Other                                    2


## Sample tables

Events by type and region, and by year and region.

In [7]:
by_type = pd.crosstab(df["Event_Type"], df["Region"], margins=True, margins_name="Total")
by_year = pd.crosstab(df["Date"].dt.year.rename("Year"), df["Region"], margins=True, margins_name="Total")
by_type.to_csv(TABLES / "events_by_type_region.csv")
by_year.to_csv(TABLES / "events_by_year_region.csv")
print(by_type.to_string())
by_year

Region      EU   US  Total
Event_Type                
CM          25  219    244
CP          31  229    260
CS           8   71     79
MA           6   19     25
Other        0    7      7
Total       70  545    615


Region,EU,US,Total
Year,,,
2018,10,43,53
2019,4,23,27
2020,3,22,25
2021,10,49,59
2022,5,49,54
2023,9,91,100
2024,4,62,66
2025,15,144,159
2026,10,62,72
